In [1]:
import pandas as pd
import numpy as np
import altair as alt
alt.data_transformers.enable("vegafusion")
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap


In [2]:
df_holidays = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/raw/holidays_events.parquet', engine='pyarrow')
df_sales = pd.read_csv('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/processed/salesperdayperstore.csv', header=0)

In [5]:
df_holidays.describe()
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_holidays['date'] = pd.to_datetime(df_holidays['date'])
df_holidays.info()
df_holidays.head()
df_sales.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4   description  350 non-null    object        
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[ns](1), object(4)
memory usage: 14.1+ KB


,store_nbr,date,unit_sales
0,1,2013-01-02,7417.148
1,1,2013-01-03,5873.244
2,1,2013-01-04,5919.879
3,1,2013-01-05,6318.785
4,1,2013-01-06,2199.087


In [6]:
df_holidays

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


Add complete date range into df_sales and remove holidays outside date range

In [7]:
# Define the date range you want to ensure is covered
start_date = '2013-01-01'
end_date = '2017-08-15'
complete_date_range = pd.date_range(start=start_date, end=end_date)

# Create a DataFrame with all combinations of store_nbr and complete_date_range
stores = df_sales['store_nbr'].unique()

df_dates = pd.DataFrame({
    'date': pd.concat([pd.Series(complete_date_range)] * len(stores), ignore_index=True),
    'store_nbr': sorted(list(stores) * len(complete_date_range))
})

# Merge df_dates with the original data
df_merged = pd.merge(df_dates, df_sales, on=['store_nbr', 'date'], how='left')

df_merged.fillna({'unit_sales': 0}, inplace = True)

df_merged

# Select one store
df_store_1 = df_merged[df_merged['store_nbr'] == 1].copy()

In [12]:
def weekday_weeknumber(df): 
    df = df.copy()
    # Add column with name weekday, weeknumbers
    df['weekday'] = df['date'].dt.dayofweek + 1
    df['week_nbr'] = df['date'].dt.isocalendar().week
    # Add cumulative week numbers, first define first date and first monday
    first_date = df['date'].min()
    day_of_week = first_date.weekday()
    days_to_last_monday = (day_of_week - 0 + 7) % 7
    monday_first_week = first_date - pd.Timedelta(days=days_to_last_monday)
    # Calculate cumulative week numbers and add column
    df['week_number_cum'] = (df['date'] - monday_first_week).dt.days // 7 + 1
    # Optimize memory
    df['weekday'] = df['weekday'].astype('int8')
    df['week_nbr'] = df['week_nbr'].astype('int8')
    df['week_number_cum'] = df['week_number_cum'].astype('int16')
    return df

#df_grouped = df_merged.groupby(['store_nbr', 'week_number_cum'])['unit_sales'].sum().reset_index()

#df_grouped

In [15]:
df_merged = weekday_weeknumber(df_merged)

df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91152 entries, 0 to 91151
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             91152 non-null  datetime64[ns]
 1   store_nbr        91152 non-null  int64         
 2   unit_sales       91152 non-null  float64       
 3   weekday          91152 non-null  int8          
 4   week_nbr         91152 non-null  int8          
 5   week_number_cum  91152 non-null  int16         
dtypes: datetime64[ns](1), float64(1), int16(1), int64(1), int8(2)
memory usage: 2.4 MB


In [ ]:
df_holidays = df_holidays[df_holidays['date'].isin(complete_date_range)]
df_date_range = pd.DataFrame(complete_date_range, columns=['date'])
df_holidays_normal = pd.merge(df_date_range, df_holidays, on='date', how='left')
df_holidays_normal['type'] = df_holidays_normal['type'].fillna('Normal')
df_holidays_normal['locale'] = df_holidays_normal['locale'].fillna('-')
df_holidays_normal['locale_name'] = df_holidays_normal['locale_name'].fillna('-')
df_holidays_normal['description'] = df_holidays_normal['description'].fillna('-')
df_holidays_normal['transferred'] = df_holidays_normal['transferred'].fillna(False)

df_holidays

Scaling unit_sales per store (0 to max (1))

In [ ]:
def min_max_scaling(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return series  # Avoid division by zero if all values are the same
    return (series - min_val) / (max_val - min_val)

# Apply the scaling function within each store group
df_merged['scaled_unit_sales'] = df_merged.groupby('store_nbr')['unit_sales'].transform(min_max_scaling)
df_merged = df_merged.drop(columns='unit_sales')
df_merged.head()

Eruption april and may 2016

In [ ]:
start_date = '2016-04-16'
end_date = '2016-05-16'
df_holiday_filtered = df_holidays[(df_holidays['date'] >= start_date) & (df_holidays['date'] <= end_date)]
df_holiday_filtered.head()

Add columns with national, regional and local holiday (True/False)

In [ ]:
df_national = df_holidays[df_holidays['locale'] == 'National']
df_regional = df_holidays[df_holidays['locale'] == 'Regional']
df_local = df_holidays[df_holidays['locale'] == 'Local']
l_nationaldates = df_national['date'].tolist()
l_regionaldates = df_regional['date'].tolist()
l_localdates = df_local['date'].tolist()
df_merged['national_holiday'] = df_merged['date'].isin(l_nationaldates)
df_merged['regional_holiday'] = df_merged['date'].isin(l_regionaldates)
df_merged['local_holiday'] = df_merged['date'].isin(l_localdates)
df_merged['any_holiday'] = df_merged[['national_holiday', 'regional_holiday', 'local_holiday']].any(axis=1)
df_merged['closed_holiday'] = (df_merged['any_holiday'] == True) & (df_merged['scaled_unit_sales'] == 0)
df_merged

In [ ]:
df_merged2 = pd.merge(df_merged, df_holidays[['date', 'locale']], on='date', how='left')
df_merged2['locale'] = df_merged2['locale'].fillna('False')

# Drop duplicates based on custom priority
def custom_drop_duplicates(df):
    # Sort by locale priority
    priority = {'National': 1, 'Regional': 2, 'Local': 3, 'False': 4}
    df['priority'] = df['locale'].map(priority)

    # Sort by store_nbr, date, and priority
    df = df.sort_values(by=['store_nbr', 'date', 'priority'])

    # Drop duplicates while keeping the first occurrence (the highest priority due to sorting)
    df = df.drop_duplicates(subset=['store_nbr', 'date'], keep='first')

    # Drop the temporary priority column
    df = df.drop(columns=['priority'])

    return df
df_merged2 = custom_drop_duplicates(df_merged2)


# select 1 year
start_date = '2016-01-01'
end_date = '2016-12-31'
df_merged2 = df_merged2[(df_merged2['date'] >= start_date) & (df_merged2['date'] <= end_date)]

df_merged2

In [ ]:
# Convert 'date' column to object type
df_merged2['date'] = df_merged2['date'].astype('object')

l_stores = df_merged2['store_nbr'].unique().tolist()

color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
                 '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78',
                 '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7',
                 '#dbdb8d', '#9edae5']


for store in l_stores:
    df_store = df_merged2[df_merged2['store_nbr'] == store]
    
    # Define color scale for locale
    unique_locales = df_store['locale'].unique()
    color_scale = alt.Scale(domain=unique_locales, range=color_palette[:len(unique_locales)])

    # Create the Altair chart
    hist = alt.Chart(df_store).mark_bar().encode(
        x=alt.X('date:T', title='Date'),
        y=alt.Y('scaled_unit_sales:Q', title='Unit Sales'),
        color=alt.Color('locale:N', scale=color_scale, title='Locale'),
        tooltip=['date', 'scaled_unit_sales', 'locale']
    ).properties(
        title=f'Total sales per day for store {store}, colored by regionality',
        width=1250,
        height=200
    )

    hist.display()

In [ ]:
df1 = df_holidays_normal
df2 = df_merged
df2_pivot = df2.pivot(index='date', columns='store_nbr', values='scaled_unit_sales').reset_index()
df_holidaysales = pd.merge(df1, df2_pivot, on='date', how='left')
df_holidaysales.columns = df_holidaysales.columns.astype(str)

# Combine the first 5 columns into one string with ' | ' separator
df_holidaysales['combined'] = df_holidaysales[['date', 'type', 'locale', 'locale_name', 'description']].apply(lambda row: ' | '.join(row.values.astype(str)), axis=1)

# Optional: drop the original 5 columns if you only need the combined column
df_holidaysales = df_holidaysales.drop(columns=['transferred', 'date', 'type', 'locale', 'locale_name', 'description'])

column_order = ['combined'] + [col for col in df_holidaysales.columns if col != 'combined']
df_holidaysales = df_holidaysales[column_order]

# Set 'combined' as the index for better heatmap visualization
df_holidaysales.set_index('combined', inplace=True)

colors = ['red'] + plt.cm.YlGnBu(np.linspace(0, 1, 256)).tolist()  # Red for zero, YlGnBu for non-zero
cmap = ListedColormap(colors)

# Create a mask for zero values
mask = (df_holidaysales == 0.00)

# Plotting the heatmap without numbers
plt.figure(figsize=(18, 400))  # Adjust the size as needed
sns.heatmap(df_holidaysales, mask=mask, cmap='YlGnBu', annot=False, linewidths=0.5, linecolor='black', cbar=False)
sns.heatmap(df_holidaysales, mask=~mask, cmap=cmap, annot=False, linewidths=0.5, linecolor='black', cbar=False)
plt.title('Heatmap of Values for Each Combined Entry')
plt.xlabel('Columns')
plt.ylabel('Combined Entries')
plt.xticks(rotation=45)  # Rotate column labels for better readability
plt.yticks(rotation=0)  # Keep row labels horizontal
plt.tight_layout()  # Adjust layout to fit everything nicely

# Show the plot
plt.show()